<a href="https://colab.research.google.com/github/denmex/CIS3120/blob/main/CIS3120_Module14_AI_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 14 — Generative AI Web Services

**Course:** CIS 3120 — Programming for Analytics  
**Section:** BMWB · Spring 2026  
  
**Prerequisite:** Module 13 — Web APIs and Authentication  

---

## How to Use This Notebook

This notebook accompanies the Module 14 reading page. The narrative explanations are kept short here; for the complete treatment of each topic, refer to the reading page. The notebook focuses on runnable code: setup cells, worked examples for both providers, and the nine scaffolded exercises (A1–A4 and B1–B5).

The notebook is divided into five sections that mirror the reading page:

1. **Generative AI Introduction** — vocabulary recap.
2. **Generative AI Use Cases** — categories of value.
3. **Generative AI Web Services** — what API access enables.
4. **Practical Application of Generative AI Web Services** — runnable examples for OpenAI and Anthropic, followed by the scaffolded exercises.
5. **Summary and Next Steps** — comparison and forward pointers.

Cells fall into two categories:

- **Pre-implemented cells.** Setup, basic API calls, helpers, and worked examples are complete and runnable. Read each one, then run it.
- **Scaffolded TODO cells.** Each exercise contains a function or loop with `TODO` markers. The structural skeleton is provided; the student writes the prompt strings, chooses the test inputs, and fills in the function bodies.

---

## Before You Begin

Two API keys are required: one from OpenAI and one from Anthropic. Both providers charge per token. Set a hard spending cap on each provider's billing page before running this notebook. Classroom usage typically costs a few cents per student; a runaway loop can cost more.

Add the keys to Google Colab Secrets:

- Click the key icon in the left sidebar.
- Add a secret named `OPENAI_API_KEY` with your OpenAI key as the value.
- Add a secret named `ANTHROPIC_API_KEY` with your Anthropic key as the value.
- Enable notebook access for both secrets.

Never paste an API key into a code cell. Never commit a notebook that contains a key. The Colab Secrets pattern was established in Module 13 and is carried forward unchanged.

---

**Shortlink:** `bit.ly/cis3120_module14`

## I. Generative AI Introduction

### Core Vocabulary (Recap)

The following terms are used throughout this notebook. Full definitions are in the reading page.

- **Generative AI** — a class of machine learning systems that produce new content rather than only classifying or predicting.
- **Large language model (LLM)** — a neural network trained on text that generates fluent text in response to a textual input.
- **Token** — the unit of text an LLM processes. As an English approximation, one token is roughly four characters or three-quarters of a word.
- **Prompt** — the textual input supplied to the model.
- **Completion** (or **response**) — the text produced by the model.
- **Context window** — the maximum total tokens (input plus output) the model can process in a single call.
- **Temperature** — a numeric parameter, typically between 0 and 1, controlling output randomness. Use 0 for analytical and extractive work; use higher values for drafting.
- **System prompt** (or **instructions**) — standing instructions provided to the model that govern its behavior throughout the conversation.
- **Hallucination** — confident-sounding text from the model that is factually incorrect.
- **Retrieval-augmented generation (RAG)** — a pattern in which relevant document passages are retrieved and supplied to the model as context.

A model has no memory between separate API calls. Conversational continuity is supplied by the caller, who passes the prior turns back in each new request.

## II. Generative AI Use Cases

The reading page organizes use cases by the type of value created. The categories appear repeatedly in the exercises:

- **Summarization** — condensing long documents into shorter summaries (Exercises A1, B1).
- **Classification and sentiment analysis** — assigning text to categories (Exercises A2, B2).
- **Information extraction** — pulling structured fields from unstructured text (Exercises A3, B3).
- **Drafting and content generation** — producing first drafts that humans then edit.
- **Translation and style transfer** — rewriting between languages or registers.
- **Code generation and explanation** — producing, translating, or explaining code.
- **Question answering over documents** — combining retrieval with generation (RAG).
- **Conversational interfaces** — multi-turn chatbots and assistants (Exercises A4, B5).

Where generative AI adds less value: exact arithmetic, access to live authoritative data without retrieval, decisions with regulatory or safety implications, and outputs whose correctness is hard to verify.

## III. Generative AI Web Services

API access supplies the same models used through the web interface, but through a programmatic interface that supports automation, integration, reproducibility, customization through system prompts, and programmatic control of output format.

Three operational concerns apply to every generative AI API:

- **Cost.** Providers charge per token. Output tokens are typically more expensive than input tokens. A short conversation costs fractions of a cent; a large batch job can cost meaningful sums.
- **Authentication.** Each request must include an API key. Keys are loaded from Colab Secrets in this notebook. Keys must never be embedded in source code or committed to a Git repository.
- **Rate limits.** Providers cap requests per minute and tokens per minute. Exceeding the cap produces an HTTP 429 error. Production code handles this with retry logic.

The API is **stateless with respect to conversational memory**. The server does not remember prior turns. Multi-turn conversations require the caller to pass the full conversation history on every request.

## IV. Practical Application of Generative AI Web Services

The remainder of this notebook is hands-on. Section IV.A covers OpenAI; Section IV.B covers Anthropic. Both follow the same pattern: account setup, key loading, a basic call, options (system prompt, temperature, multi-turn, JSON, token usage), and four or five scaffolded exercises.

### A Shared Output Helper

The cell below defines `show()`, a small helper that prints a labeled response and (optionally) token counts. It is used throughout this section so that worked examples and exercises produce consistent output.

In [ ]:
def show(label: str, text: str, usage: dict | None = None) -> None:
    """Print a labeled response with an optional usage summary.

    Parameters
    ----------
    label : str
        A short label for the response, for example "OpenAI" or "Anthropic".
    text : str
        The text body of the response.
    usage : dict, optional
        A dictionary with keys "input" and "output" containing token counts.
    """
    print(f"--- {label} " + "-" * (60 - len(label)))
    print(text)
    if usage is not None:
        print()
        print(f"Input tokens : {usage['input']}")
        print(f"Output tokens: {usage['output']}")
        print(f"Total tokens : {usage['input'] + usage['output']}")
    print()

---

## IV. A. OpenAI

OpenAI's API exposes several model families. This notebook uses the **Chat Completions API**, whose message-list structure mirrors the Anthropic Messages API and supports direct A/B comparison between providers.

### Step 1 — Install the OpenAI Python SDK

In [ ]:
!pip install --quiet openai

### Step 2 — Load the API key from Colab Secrets and instantiate the client

The `OpenAI()` constructor reads the API key from the `OPENAI_API_KEY` environment variable, so no further configuration is required.

In [ ]:
from google.colab import userdata
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
openai_client = OpenAI()

### A Basic Example

The following call sends a single user message and prints the model's response. The `messages` argument is a list of message objects, each with a `role` and `content` field. The reply is found at `response.choices[0].message.content`.

The model identifier used throughout this notebook is `gpt-5.5`. If the identifier has changed since this notebook was written, substitute the current name as published in OpenAI's documentation.

In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {"role": "user", "content": "In one sentence, explain what an API is."}
    ]
)

show("OpenAI", response.choices[0].message.content)

--- OpenAI ------------------------------------------------------
An API (Application Programming Interface) is a set of rules that lets different software applications communicate and share data or functionality with each other.



### Adding Options and Complexity

#### System message

Prepending a message with the role `system` supplies standing instructions that shape every response.

In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {"role": "system", "content": "You are a financial analyst. "
                                       "Respond in plain language suitable for a non-technical "
                                       "audience. Limit responses to three sentences."},
        {"role": "user", "content": "What is the difference between revenue and profit?"}
    ]
)

show("OpenAI · system message", response.choices[0].message.content)

--- OpenAI · system message -------------------------------------
Revenue is the total money a business brings in from selling its products or services.  
Profit is what remains after the business subtracts its costs and expenses from that revenue.  
For example, if a company earns $100,000 in sales and spends $70,000 to operate, its profit is $30,000.



In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {"role": "system", "content": "You are a stand-up comedian. "
                                       "Respond as if you are the entertainer an an accounting conference. "
                                       "Show the audience the futility of their work."},
        {"role": "user", "content": "What is the difference between revenue and profit?"}
    ]
)

show("OpenAI · system message", response.choices[0].message.content)

--- OpenAI · system message -------------------------------------
Revenue is all the money a company brings in.

Profit is what’s left after the company pays for everything it needed to make that revenue: salaries, rent, software, inventory, taxes, “team-building” retreats nobody asked for, and one mysterious consulting invoice from a guy named Brad.

So:

**Revenue = money coming in**  
**Profit = money left over**

Example:

A company sells **$1,000,000** worth of products. Great news — huge revenue! Everyone claps. The CEO buys a vest.

But then expenses are:

- Cost of goods: $400,000  
- Payroll: $300,000  
- Rent: $100,000  
- Marketing: $150,000  
- Miscellaneous “strategic initiatives”: $75,000  

Now the company has **negative profit** — also known as “a startup.”

So revenue is the big impressive number companies brag about.

Profit is the smaller, more honest number accountants find after ruining the party.

In short: **Revenue is the applause. Profit is the paycheck.**



#### Temperature

Setting `temperature=0` produces focused, near-deterministic output. Higher values (0.7 to 1.0) produce more varied output. For data extraction and classification, use 0. For drafting and creative work, use higher values.

In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-5.5",
    # temperature=0, # gpt-5.5 only uses a temperature of 1, older models are flexible
    messages=[
        {"role": "user", "content": "Extract the company name and the dollar amount from this sentence: "
                                     "Acme Corporation reported quarterly revenue of $42.7 million."}
    ]
)

show("OpenAI · temperature 0", response.choices[0].message.content)

#### Multi-turn conversation

To carry a conversation across turns, append each user message and each assistant response to the messages list and resend the full list with each new request.

In [ ]:
messages = [
    {"role": "system", "content": "You are a Python tutor. Answer concisely."}
]

# Turn 1
messages.append({"role": "user", "content": "What is a list comprehension?"})
response = openai_client.chat.completions.create(model="gpt-5.5", messages=messages)
reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": reply})
show("OpenAI · turn 1", reply)

# Turn 2 — the model has access to the prior turn
messages.append({"role": "user", "content": "Can you give a small example?"})
response = openai_client.chat.completions.create(model="gpt-5.5", messages=messages)
reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": reply})
show("OpenAI · turn 2", reply)

#### Structured output as JSON

When downstream code needs to parse the response, ask explicitly for JSON in the prompt and pass `response_format={"type": "json_object"}` for higher reliability. Then parse the result with `json.loads()`.

In [ ]:
import json

prompt = """Extract the following fields from the text below and return them as JSON
with keys "company", "quarter", "revenue_millions":

Text: Acme Corporation reported quarterly revenue of $42.7 million in Q3 2025."""

response = openai_client.chat.completions.create(
    model="gpt-5.5",
    # temperature=0, # gpt-5.5 only uses a temperature of 1, older models are flexible
    response_format={"type": "json_object"},
    messages=[{"role": "user", "content": prompt}]
)

data = json.loads(response.choices[0].message.content)
print(data)
print("Revenue (millions):", data["revenue_millions"])

#### Inspecting token usage

Every response includes a `usage` object that reports input and output token counts. This is the basis for cost tracking.

In [ ]:
print("Input tokens : ", response.usage.prompt_tokens)
print("Output tokens: ", response.usage.completion_tokens)
print("Total tokens : ", response.usage.total_tokens)

---

### Scaffolded Exercises — OpenAI

Each exercise below builds on the previous one. Complete them in order. Each is intentionally underspecified in places where the student should make and document a design choice. The skeletons are provided; the prompt strings, test inputs, and function bodies are the student's work.

#### Exercise A1 — Two-line summary

Write a function `summarize(text)` that takes a string of any length and returns a two-sentence summary using the OpenAI API. Test it on a paragraph from any news article. Use `temperature=0`.

**Hint:** place the summarization instruction in a system message and the article text in a user message.

In [ ]:
def summarize(text: str) -> str:
    """Return a two-sentence summary of `text` using the OpenAI API."""
    # TODO 1: write a clear system message that instructs the model to produce
    #         a summary of exactly two sentences. Be explicit about the count.
    system_message = ""  # TODO

    # TODO 2: build the messages list with the system message and the
    #         article text as a user message.
    messages = []  # TODO

    # TODO 3: call openai_client.chat.completions.create with the model,
    #         temperature=0, and the messages list. Return the reply text.
    pass  # TODO


# Test the function on a real paragraph. Paste a paragraph from any news
# article into `article` below. Two to four sentences is plenty.
article = """
TODO — paste a paragraph from any news article here.
"""

summary = summarize(article)
show("OpenAI · A1 summary", summary)

#### Exercise A2 — Sentiment classifier

Write a function `classify_sentiment(review)` that returns one of the strings `"positive"`, `"negative"`, or `"neutral"`. Test it on five product reviews of your choosing — two clearly positive, two clearly negative, one ambiguous. Report which one the model finds hardest.

**Hint:** instruct the model to return only one of the three words and nothing else. Use `temperature=0`.

In [ ]:
def classify_sentiment(review: str) -> str:
    """Return one of: "positive", "negative", "neutral"."""
    # TODO 1: write a system message that:
    #         - tells the model it is classifying product-review sentiment
    #         - lists the three allowed output strings exactly
    #         - instructs the model to return ONLY one of those strings
    #           with no surrounding explanation or punctuation.
    system_message = ""  # TODO

    # TODO 2: build the messages list and call the API at temperature=0.
    # TODO 3: return the reply, stripped of whitespace and lower-cased.
    pass  # TODO


# TODO: replace the placeholders with five reviews of your choosing.
#       Two clearly positive, two clearly negative, one ambiguous.
reviews = [
    "TODO — clearly positive review #1",
    "TODO — clearly positive review #2",
    "TODO — clearly negative review #1",
    "TODO — clearly negative review #2",
    "TODO — deliberately ambiguous review",
]

for i, r in enumerate(reviews, start=1):
    label = classify_sentiment(r)
    print(f"Review {i} ({label}): {r[:60]}...")

# After running, write a short note in the cell below identifying which
# review the model classified incorrectly or with the lowest apparent
# confidence, and explain why.

*TODO: Write a one-sentence observation about which review the model finds hardest and why.*

#### Exercise A3 — Structured extraction

Choose three short news articles about earnings announcements. Write a function `extract_earnings(article)` that returns a Python dictionary with the keys `company_name`, `period`, `revenue`, and `net_income`. Use the `response_format` parameter to request JSON. Print the resulting dictionary for each article.

**Hint:** in the prompt, state explicitly that any unknown field should be returned as `null`.

In [ ]:
import json

def extract_earnings(article: str) -> dict:
    """Extract earnings fields from `article` and return them as a dict.

    Returned keys:
        company_name : str or None
        period       : str or None  (for example, "Q3 2025")
        revenue      : str or None  (the raw revenue figure including unit)
        net_income   : str or None  (the raw net income figure including unit)
    """
    # TODO 1: write a prompt that:
    #         - names the four required keys exactly
    #         - instructs the model to return ONLY a JSON object
    #         - tells the model to use null for any field it cannot find
    prompt = ""  # TODO

    # TODO 2: call openai_client.chat.completions.create with:
    #         - model='gpt-5.5'
    #         - temperature=0
    #         - response_format={"type": "json_object"}
    #         - messages = a single user message containing the prompt
    # TODO 3: parse the response with json.loads() and return the dictionary.
    pass  # TODO


# TODO: replace the placeholders with three short earnings-announcement
#       articles. One sentence is enough; a paragraph is better.
articles = [
    "TODO — earnings article #1",
    "TODO — earnings article #2",
    "TODO — earnings article #3",
]

for i, a in enumerate(articles, start=1):
    print(f"--- Article {i} ---")
    print(extract_earnings(a))
    print()

#### Exercise A4 — Mini chatbot

Build an interactive loop in Colab that maintains a conversation with the model across multiple turns. The user types a message, the model responds, and the loop continues until the user types `quit`. Add a system message that gives the assistant a specific role of your choosing — for example, a Python tutor, a recipe assistant, or a study coach.

**Hint:** use `input()` to read user messages. Maintain a `messages` list and append both user and assistant messages to it on each turn. Print the running token total after each turn so you can observe how multi-turn conversations grow.

In [ ]:
def openai_chatbot() -> None:
    """Run an interactive multi-turn chat loop against the OpenAI API."""
    # TODO 1: pick an assistant role and write the system message.
    #         Examples: a Python tutor, a recipe assistant, a study coach.
    system_message = ""  # TODO

    messages = [{"role": "system", "content": system_message}]
    total_tokens = 0

    print("Type your message and press Enter. Type 'quit' to exit.\n")

    while True:
        user_text = input("You: ").strip()
        if user_text.lower() == "quit":
            print(f"\nConversation ended. Total tokens used: {total_tokens}")
            break

        # TODO 2: append the user message to `messages`.
        # TODO 3: call openai_client.chat.completions.create with the
        #         current `messages` list and the model identifier.
        # TODO 4: extract the assistant reply, append it to `messages`
        #         as an assistant message, and print it labeled "Assistant:".
        # TODO 5: add response.usage.total_tokens to `total_tokens` and
        #         print the running total after each turn.
        pass  # TODO


# Run the chatbot. To stop, type quit.
openai_chatbot()

---

## IV. B. Anthropic

Anthropic's API is built around a single primary endpoint, the **Messages API**. The procedure parallels OpenAI's, with three differences that must be observed:

1. The `max_tokens` parameter is **required**. The call fails without it.
2. The system prompt is a separate top-level `system=` parameter rather than an entry in the `messages` list.
3. The response content is a typed list. The text is at `message.content[0].text`, not `.choices[0].message.content`.

### Step 1 — Install the Anthropic Python SDK

In [ ]:
!pip install --quiet anthropic

### Step 2 — Load the API key from Colab Secrets and instantiate the client

In [ ]:
from google.colab import userdata
import os
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
anthropic_client = anthropic.Anthropic()

### A Basic Example

The model identifier used throughout this notebook is `claude-opus-4-7`.

In [ ]:
message = anthropic_client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "In one sentence, explain what an API is."}
    ]
)

show("Anthropic", message.content[0].text)

### Adding Options and Complexity

#### System prompt

Anthropic places the system prompt in a separate top-level parameter rather than in the messages list.

In [ ]:
message = anthropic_client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    system="You are a financial analyst. Respond in plain language "
           "suitable for a non-technical audience. Limit responses to three sentences.",
    messages=[
        {"role": "user", "content": "What is the difference between revenue and profit?"}
    ]
)

show("Anthropic · system prompt", message.content[0].text)

#### Temperature

Anthropic accepts the same `temperature` parameter, ranging from 0 to 1.

In [ ]:
message = anthropic_client.messages.create(
    model="claude-opus-4-7",
    max_tokens=256,
    # temperature=0,  # temperature is deprecated for the claude-opus-4-7 model
    messages=[
        {"role": "user", "content": "Extract the company name and the dollar amount from this sentence: "
                                     "Acme Corporation reported quarterly revenue of $42.7 million."}
    ]
)

show("Anthropic · temperature deprecated", message.content[0].text) # temperature is deprecated for the claude-opus-4-7 model

#### Multi-turn conversation

As with OpenAI, the caller maintains the conversation history and resends it on every turn. The system prompt is passed on every call but is not appended to `messages`.

In [ ]:
messages = []

# Turn 1
messages.append({"role": "user", "content": "What is a list comprehension?"})
response = anthropic_client.messages.create(
    model="claude-opus-4-7",
    max_tokens=512,
    system="You are a Python tutor. Answer concisely.",
    messages=messages
)
reply = response.content[0].text
messages.append({"role": "assistant", "content": reply})
show("Anthropic · turn 1", reply)

# Turn 2
messages.append({"role": "user", "content": "Can you give a small example?"})
response = anthropic_client.messages.create(
    model="claude-opus-4-7",
    max_tokens=512,
    system="You are a Python tutor. Answer concisely.",
    messages=messages
)
reply = response.content[0].text
messages.append({"role": "assistant", "content": reply})
show("Anthropic · turn 2", reply)

#### Structured output as JSON

Anthropic does not currently use a separate `response_format` parameter for JSON. The standard pattern is to request JSON in the prompt and parse the result. For greater reliability, ask the model to return only the JSON with no surrounding prose.

In [ ]:
import json

prompt = """Extract the following fields from the text below and return them as JSON
with keys "company", "quarter", "revenue_millions". Return only the JSON,
no other text.

Text: Acme Corporation reported quarterly revenue of $42.7 million in Q3 2025."""

message = anthropic_client.messages.create(
    model="claude-opus-4-7",
    max_tokens=256,
    # temperature=0,  # temperature is deprecated for the claude-opus-4-7 model
    messages=[{"role": "user", "content": prompt}]
)

data = json.loads(message.content[0].text)
print(data)
print("Revenue (millions):", data["revenue_millions"])   # temperature is deprecated for the claude-opus-4-7 model

#### Inspecting token usage

The Anthropic response includes a `usage` object reporting input and output token counts.

In [ ]:
print("Input tokens : ", message.usage.input_tokens)
print("Output tokens: ", message.usage.output_tokens)

#### Choosing a model tier

For high-volume, simple workloads such as classification, the Haiku tier offers the lowest cost and the fastest response. For most analytical and drafting work, Sonnet is the standard choice. For the most demanding reasoning, summarization of long documents, or code generation work, Opus is appropriate. Confirm current identifiers in the Anthropic documentation.

---

### Scaffolded Exercises — Anthropic

These exercises mirror the OpenAI exercises in structure to permit direct comparison. Complete them in order.

#### Exercise B1 — Two-line summary

Write a function `summarize_with_claude(text)` that takes a string and returns a two-sentence summary using the Anthropic API. Test it on the same article you used for Exercise A1 and compare the two summaries. Note any differences in style, length, or emphasis.

**Hint:** place the summarization instruction in the `system` parameter and the article text as a user message.

In [ ]:
def summarize_with_claude(text: str) -> str:
    """Return a two-sentence summary of `text` using the Anthropic API."""
    # TODO 1: write the summarization instruction. Note that for Anthropic
    #         this goes in the `system=` parameter, NOT in the messages list.
    system_instruction = ""  # TODO

    # TODO 2: call anthropic_client.messages.create with:
    #         - model='claude-opus-4-7'
    #         - max_tokens (a reasonable cap; remember it is required)
    #         - system=system_instruction
    #         - messages=[{"role": "user", "content": text}]
    # TODO 3: return message.content[0].text
    pass  # TODO


# TODO: paste the SAME article text used in Exercise A1 so the two summaries
#       can be compared directly.
article = """
TODO — paste the same article paragraph from Exercise A1 here.
"""

summary = summarize_with_claude(article)
show("Anthropic · B1 summary", summary)

*TODO: Write two or three sentences comparing this summary to the one produced in Exercise A1. Note differences in length, style, emphasis, or any factual handling.*

#### Exercise B2 — Sentiment classifier with confidence

Write a function `classify_with_confidence(review)` that returns a Python dictionary with two keys: `sentiment` (one of `"positive"`, `"negative"`, or `"neutral"`) and `confidence` (a number from 0 to 1). Test on the same five reviews used in Exercise A2. Compare the two classifiers' outputs.

**Hint:** instruct the model to return JSON only, with the two specified keys. Use `temperature=0`. Parse the result with `json.loads()`.

In [ ]:
import json

def classify_with_confidence(review: str) -> dict:
    """Classify sentiment and return {"sentiment": ..., "confidence": ...}."""
    # TODO 1: write a prompt that:
    #         - lists the three allowed sentiment values exactly
    #         - asks for a confidence number between 0 and 1
    #         - instructs the model to return ONLY a JSON object
    #           with keys "sentiment" and "confidence" and nothing else.
    prompt = ""  # TODO

    # TODO 2: call anthropic_client.messages.create with model, max_tokens,
    #         temperature=0, and a single user message containing the prompt.
    # TODO 3: parse the response text with json.loads() and return the dict.
    pass  # TODO


# TODO: reuse the SAME five reviews from Exercise A2.
reviews = [
    "TODO — review #1 (same as A2)",
    "TODO — review #2 (same as A2)",
    "TODO — review #3 (same as A2)",
    "TODO — review #4 (same as A2)",
    "TODO — review #5 (same as A2)",
]

for i, r in enumerate(reviews, start=1):
    result = classify_with_confidence(r)
    print(f"Review {i}: {result}")

# After running, compare these results with the OpenAI results from A2.
# Did the two providers agree on every review? Where they disagreed, which
# answer seems more reasonable?

*TODO: Write two or three sentences comparing the OpenAI (A2) and Anthropic (B2) classifications. Where did they agree, where did they disagree, and which output do you find more useful?*

#### Exercise B3 — Structured extraction with multiple records

Choose a single news article that mentions multiple companies and their financial figures. Write a function `extract_all_earnings(article)` that returns a Python list of dictionaries, one per company, with the keys `company_name`, `period`, `revenue`, and `net_income`.

**Hint:** instruct the model to return a JSON array. Be explicit in the prompt that any unknown field should be returned as `null` and that companies mentioned without financial figures should still appear in the array.

In [ ]:
import json

def extract_all_earnings(article: str) -> list:
    """Return a list of dicts, one per company mentioned in `article`."""
    # TODO 1: write a prompt that:
    #         - lists the four keys per record exactly
    #         - instructs the model to return ONLY a JSON array
    #         - tells the model to use null for any unknown field
    #         - tells the model to include companies even if no financial
    #           figures are given for them.
    prompt = ""  # TODO

    # TODO 2: call anthropic_client.messages.create with model, max_tokens,
    #         temperature=0, and the prompt as a user message.
    # TODO 3: parse the response with json.loads() and return the list.
    pass  # TODO


# TODO: paste a news article that mentions at least two companies and their
#       financial figures. A short paragraph is sufficient.
article = """
TODO — paste a multi-company earnings article here.
"""

records = extract_all_earnings(article)
for r in records:
    print(r)

#### Exercise B4 — Comparative document review

Pick two short documents on the same topic — two product reviews, two news articles, two policy statements. Write a function `compare_documents(doc_a, doc_b)` that returns a structured comparison containing: areas of agreement, areas of disagreement, and a recommendation about which document is more credible and why.

**Hint:** provide both documents in a single user message with clear delimiters. Use the `system` parameter to establish the model as an analytical reviewer. Allow a higher `max_tokens` value (around 2000) to permit a thorough response.

In [ ]:
def compare_documents(doc_a: str, doc_b: str) -> str:
    """Return a structured comparison of two documents on the same topic."""
    # TODO 1: write a system instruction that establishes the model as an
    #         analytical reviewer producing structured comparisons.
    system_instruction = ""  # TODO

    # TODO 2: build a user message that contains both documents with clear
    #         delimiters. For example:
    #             === DOCUMENT A ===
    #             ... text of doc_a ...
    #             === DOCUMENT B ===
    #             ... text of doc_b ...
    #         Then ask for three sections in the reply:
    #         areas of agreement, areas of disagreement, credibility verdict.
    user_message = ""  # TODO

    # TODO 3: call anthropic_client.messages.create with:
    #         - model='claude-opus-4-7'
    #         - max_tokens=2000   (B4 expects a longer response)
    #         - system=system_instruction
    #         - messages=[{"role": "user", "content": user_message}]
    # TODO 4: return message.content[0].text
    pass  # TODO


# TODO: paste two short documents on the same topic.
doc_a = """
TODO — document A.
"""

doc_b = """
TODO — document B (same topic, different source or perspective).
"""

result = compare_documents(doc_a, doc_b)
show("Anthropic · B4 comparison", result)

#### Exercise B5 — Build a comparable mini chatbot

Repeat Exercise A4 using the Anthropic API instead of OpenAI. Then write a brief paragraph comparing the two experiences — differences in the SDK shape, response style, and any other observations.

**Hint:** the conversation-management logic is identical; only the SDK calls and the response-extraction syntax differ. Recall that for Anthropic the system prompt is a separate parameter (not a message), `max_tokens` is required, and the reply is at `message.content[0].text`.

In [ ]:
def anthropic_chatbot() -> None:
    """Run an interactive multi-turn chat loop against the Anthropic API."""
    # TODO 1: pick an assistant role (use the same role you chose in A4 so
    #         the two chatbots can be compared on equal footing).
    system_instruction = ""  # TODO

    messages = []  # NOTE: for Anthropic, the system prompt is NOT in here.
    total_input_tokens = 0
    total_output_tokens = 0

    print("Type your message and press Enter. Type 'quit' to exit.\n")

    while True:
        user_text = input("You: ").strip()
        if user_text.lower() == "quit":
            total = total_input_tokens + total_output_tokens
            print(f"\nConversation ended. Total tokens used: {total} "
                  f"(input {total_input_tokens}, output {total_output_tokens})")
            break

        # TODO 2: append the user message to `messages`.
        # TODO 3: call anthropic_client.messages.create with:
        #         - model='claude-opus-4-7'
        #         - max_tokens (a reasonable cap)
        #         - system=system_instruction
        #         - messages=messages
        # TODO 4: extract reply with response.content[0].text, append it to
        #         `messages` as an assistant message, print it labeled
        #         "Assistant:".
        # TODO 5: add response.usage.input_tokens and
        #         response.usage.output_tokens to the running totals and
        #         print them after each turn.
        pass  # TODO


# Run the chatbot. To stop, type quit.
anthropic_chatbot()

*TODO: Write three to five sentences comparing the OpenAI chatbot (A4) and the Anthropic chatbot (B5). Comment on differences in SDK shape, where the system prompt lives, response style, response length, and any other observations.*

---

## V. Summary and Next Steps

### What Has Been Covered

This module introduced the conceptual foundation of generative AI and large language models, distinguished API access from web-interface access, and walked through the practical use of two providers' APIs from Google Colab. The same nine-exercise scaffold (A1–A4 and B1–B5) was completed against both OpenAI and Anthropic, supporting direct A/B comparison.

### Comparing the Two Providers

| Aspect | OpenAI | Anthropic |
|---|---|---|
| API used here | Chat Completions | Messages |
| System prompt | Entry in `messages` list | Top-level `system=` parameter |
| `max_tokens` | Optional | **Required** |
| Response text location | `response.choices[0].message.content` | `message.content[0].text` |
| JSON output | `response_format={"type": "json_object"}` plus prompt | Prompt instruction only |
| Token usage | `prompt_tokens`, `completion_tokens`, `total_tokens` | `input_tokens`, `output_tokens` |

Neither provider is universally better. Both are first-tier; both improve frequently. The right choice depends on the specific workload, cost, latency requirements, and any provider-specific features the application needs.

### Suggested Next Topics

The reading page lists seven topics that build on this module's foundation. They are out of scope for Module 14 but worth knowing about:

1. **Embeddings and semantic search.**
2. **Retrieval-augmented generation (RAG).**
3. **Tool use (function calling).**
4. **Prompt engineering.**
5. **Evaluation.**
6. **Cost and latency optimization.**
7. **Orchestration frameworks.** LangChain is the most widely adopted Python framework for orchestrating multi-step LLM pipelines, retrieval-augmented generation, and tool-using agents. It reached its first stable v1.0 release in October 2025 and is paired with LangGraph for agent runtime and LangSmith for observability. The native SDK foundation taught in this module is the prerequisite for using such frameworks effectively.

---

## Reference Documentation

- **OpenAI API documentation:** https://platform.openai.com/docs
- **Anthropic API documentation:** https://docs.claude.com
- **PY4E Chapter 13 — Web APIs (prerequisite):** https://www.py4e.com/html3/13-web

Both provider documentation sites are updated frequently. Confirm the current model identifiers, parameter names, and pricing at the start of any new project.

---

*End of Module 14 Student Notebook.*